# Temporal Squared Gap Length Index Evaluation

This notebook evaluates the performance of NuFrost, Zhu2015, and HANTS algorithms under different Temporal Squared Gap Length Index ($I$) values.

$$
    I_{gap} = \frac{1}{N^2} \left[ \left(\sum_i l_i \cos 2\pi\tau_i\right)^2 + \left(\sum_i l_i \sin 2\pi\tau_i\right)^2 \right]
$$

Where $N$ is the total sequence length, and $l_j$ is the length of the $j$-th continuous gap. We simulate different gap conditions by taking valid pixels and artificially dropping continuous blocks of different lengths.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../')
from src.data_loader import RSCube
from src.nufrost import timestamps_to_seconds, predict_single_pixel
from src.zhu2015 import fit_predict_pixel
from src.hants import hants_pixel
from src.evaluation import compute_metrics
from config.settings import build_args
from joblib import Parallel, delayed

sns.set_theme(style="whitegrid", font_scale=1.2, rc={"axes.edgecolor": "black", "axes.linewidth": 1})
method_palette = {'NuFrost': '#D62728', 'Zhu2015': '#1F77B4', 'HANTS': '#fbbc02'}


In [ ]:
# ---------------------------------------------------------
# Configuration settings for the Temporal Gap Evaluation
# ---------------------------------------------------------

import time

# NASA_HLS_v002_RED_lon116.3372_lat31.3563_part1-0000000512-0000000512
TARGET_LON = 116.3372
TARGET_LAT = 31.3563
TARGET_BAND = "BLUE"
HLS_DATA_DIR = "../data/hls"
IMAGE_NAMES = []
CACHE_DIR = '../data/local_cache'
OUTPUT_DIR = "../data/output"

# Lengths of continuous gaps to artificially drop
DROP_LENGTHS = [100, 300, 500, 800, 1200, 1500, 2000, 2500, 3000, 4000, 5000, 6000]

# Number of high-quality valid pixels to sample for evaluation
NUM_SAMPLES = 40000

# Random seed for reproducibility
RANDOM_SEED = time.time_ns() % (2 ** 32 - 1)  # Use current time in nanoseconds as seed

# Number of parallel jobs (-1 uses all available cores)
N_JOBS = -1


In [ ]:
def calc_gap_index(mask: np.ndarray, t_days: np.ndarray) -> float:
    """Calculate the temporal squared gap length index."""
    N = len(mask)
    if N == 0:
        return 0.0

    T_total_days = t_days[-1] - t_days[0]
    if T_total_days <= 0:
        return 0.0

    padded = np.pad(mask.astype(int), (1, 1), 'constant')
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    lengths = ends - starts

    if len(starts) == 0:
        return 0.0

    tau = np.zeros(len(starts))
    for i in range(len(starts)):
        start_day = t_days[starts[i]]
        end_idx = ends[i] - 1 if ends[i] <= N else N - 1
        end_day = t_days[end_idx]
        mid_day = (start_day + end_day) / 2.0
        tau[i] = (mid_day - t_days[0]) / T_total_days

    sum_cos = np.sum(lengths * np.cos(2 * np.pi * tau))
    sum_sin = np.sum(lengths * np.sin(2 * np.pi * tau))

    return (sum_cos**2 + sum_sin**2) / (N**2)

def evaluate_pixel_gap_lengths(t_days, t_sec, y_ts, args, drop_lengths):
    N = len(y_ts)
    base_mask = ~np.isfinite(y_ts)

    results = []
    valid_idx = np.where(~base_mask)[0]
    if len(valid_idx) < args.min_obs + 50:
        return []

    max_L = max(drop_lengths)
    L_min = min(drop_lengths)

    # Find a valid center index that allows expanding the gap symmetrically up to max_L
    valid_centers = [idx for idx in valid_idx if max_L//2 < idx < N - max_L//2]
    if not valid_centers:
        return []

    center_idx = np.random.choice(valid_centers)

    # The fixed evaluation mask ensures we measure error on the EXACT SAME points for all L
    fixed_eval_mask = np.zeros(N, dtype=bool)
    fixed_eval_mask[center_idx - L_min//2 : center_idx + L_min//2] = True
    fixed_eval_mask = fixed_eval_mask & (~base_mask)

    if not np.any(fixed_eval_mask):
        return []

    for L in drop_lengths:
        # Expand the gap symmetrically
        start_idx = max(0, center_idx - L // 2)
        end_idx = min(N, center_idx + L // 2)

        new_mask = base_mask.copy()
        new_mask[start_idx:end_idx] = True

        I_val = calc_gap_index(new_mask, t_days)

        y_corrupted = y_ts.copy()
        y_corrupted[new_mask] = np.nan

        # Use the fixed evaluation points
        y_true_eval = y_ts[fixed_eval_mask]
        t_eval_days = t_days[fixed_eval_mask]
        t_eval_secs = t_sec[fixed_eval_mask]

        preds_nufrost = []
        preds_zhu = []
        preds_hants = []

        for td, ts in zip(t_eval_days, t_eval_secs):
            # NuFrost
            pred_n, _ = predict_single_pixel(
                t_sec, y_corrupted, ts,
                nufft_modes=args.modes, eps=args.eps,
                num_peaks=args.num_peaks, power_cum=args.power_cum, ignore_dc_hz=args.ignore_dc_hz,
                refine_peaks=args.refine_peaks, include_trend=args.include_trend,
                ridge_lam=args.ridge, freq_weight=args.freq_weight, huber_iters=args.huber_iters, huber_delta=args.huber_delta,
                min_obs=args.min_obs
            )
            preds_nufrost.append(pred_n)

            # Zhu2015
            pred_z, _ = fit_predict_pixel(t_days, y_corrupted, td, lasso_alpha=0.0001)
            preds_zhu.append(pred_z)

            # HANTS
            pred_h = hants_pixel(t_days, y_corrupted, td, nof=3, sf="low", fet=0.05, dod=5)
            preds_hants.append(pred_h)

        met_n = compute_metrics(y_true_eval, np.array(preds_nufrost))
        met_z = compute_metrics(y_true_eval, np.array(preds_zhu))
        met_h = compute_metrics(y_true_eval, np.array(preds_hants))

        results.append({
            "I": I_val,
            "NuFrost_RMSE": met_n["RMSE"], "NuFrost_MAE": met_n["MAE"], "NuFrost_R": met_n["R"], "NuFrost_OutlierRatio": met_n["OutlierRatio"],
            "Zhu2015_RMSE": met_z["RMSE"], "Zhu2015_MAE": met_z["MAE"], "Zhu2015_R": met_z["R"], "Zhu2015_OutlierRatio": met_z["OutlierRatio"],
            "HANTS_RMSE": met_h["RMSE"], "HANTS_MAE": met_h["MAE"], "HANTS_R": met_h["R"], "HANTS_OutlierRatio": met_h["OutlierRatio"],
        })

    return results


In [ ]:
from src.data_loader import find_image_chunks
if not IMAGE_NAMES:
    image_paths = find_image_chunks(HLS_DATA_DIR, TARGET_LON, TARGET_LAT, TARGET_BAND)
    image_paths_list = [image_paths] if image_paths else []
else:
    image_paths_list = [[f"../data/input/{name}"] for name in IMAGE_NAMES]

print(f"Found {len(image_paths_list)} image set(s) to evaluate.")
image_paths = image_paths_list[0]  # Just take the first chunk for gap index eval
print(f"Evaluating chunk: {image_paths}")

# Load image and sample pixels
args = build_args({'image': image_paths})
loader = RSCube(image_paths, cache_dir=CACHE_DIR)
data = loader.load()
cube = data["cube"]
timestamps = data["timestamps"]

t_sec = timestamps_to_seconds(timestamps, unit="seconds")
t0_sec = np.min(t_sec)
t_days = (t_sec - t0_sec) / 86400.0

T, H, W = cube.shape
valid_counts = np.sum(np.isfinite(cube), axis=0)
valid_pixels = np.argwhere(valid_counts >= args.min_obs + 50)

np.random.seed(RANDOM_SEED)
num_samples = min(NUM_SAMPLES, len(valid_pixels))
indices = np.random.choice(len(valid_pixels), num_samples, replace=False)
sampled_pixels = valid_pixels[indices]

print(f"Selected {num_samples} pixels for evaluation.")


In [ ]:
# Run evaluation
# We drop different lengths of continuous valid observations to vary the Gap Index (I)
print("Running evaluations across different gap lengths... (This might take a few hours)")
all_results = Parallel(n_jobs=N_JOBS)(
    delayed(evaluate_pixel_gap_lengths)(t_days, t_sec, cube[:, r, c], args, DROP_LENGTHS)
    for r, c in sampled_pixels
)

flat_results = [res for pixel_res in all_results for res in pixel_res]
df_res = pd.DataFrame(flat_results)

if df_res.empty:
    print("No valid results collected.")
else:
    # Melt dataframe for easy plotting
    df_melt = []
    for metric in ['RMSE', 'MAE', 'R', 'OutlierRatio']:
        for method in ['NuFrost', 'Zhu2015', 'HANTS']:
            temp_df = df_res[['I', f'{method}_{metric}']].copy()
            temp_df.columns = ['I', 'Value']
            temp_df['Method'] = method
            temp_df['Metric'] = metric
            df_melt.append(temp_df)
    df_plot = pd.concat(df_melt, ignore_index=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, f"gap_index_evaluation_{TARGET_BAND}.csv")
    df_plot.to_csv(csv_path, index=False)
    print(f"Results saved to {csv_path}")

    # Bin I for smoother lines
    df_plot['I_bin'] = pd.cut(df_plot['I'], bins=10).apply(lambda x: x.mid).astype(float)
    print("Evaluations completed successfully!")


## 1. Root Mean Square Error (RMSE) vs Temporal Squared Gap Length Index

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_plot[df_plot['Metric'] == 'RMSE'], x='I_bin', y='Value', hue='Method', palette=method_palette, marker='o', errorbar=None, linewidth=2.5)
plt.title('RMSE vs Temporal Squared Gap Length Index (I)', pad=15, fontweight='bold')
plt.xlabel(r'Temporal Squared Gap Length Index (I) = $\frac{1}{N^2}\sum l_j^2$', fontweight='bold')
plt.ylabel('RMSE', fontweight='bold')
plt.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 2. Mean Absolute Error (MAE) vs Temporal Squared Gap Length Index

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_plot[df_plot['Metric'] == 'MAE'], x='I_bin', y='Value', hue='Method', palette=method_palette, marker='o', errorbar=None, linewidth=2.5)
plt.title('MAE vs Temporal Squared Gap Length Index (I)', pad=15, fontweight='bold')
plt.xlabel(r'Temporal Squared Gap Length Index (I) = $\frac{1}{N^2}\sum l_j^2$', fontweight='bold')
plt.ylabel('MAE', fontweight='bold')
plt.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Correlation Coefficient (R) vs Temporal Squared Gap Length Index

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_plot[df_plot['Metric'] == 'R'], x='I_bin', y='Value', hue='Method', palette=method_palette, marker='o', errorbar=None, linewidth=2.5)
plt.title('R vs Temporal Squared Gap Length Index (I)', pad=15, fontweight='bold')
plt.xlabel(r'Temporal Squared Gap Length Index (I) = $\frac{1}{N^2}\sum l_j^2$', fontweight='bold')
plt.ylabel('Correlation Coefficient (R)', fontweight='bold')
plt.ylim(0, 1.05)
plt.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Outlier Ratio vs Temporal Squared Gap Length Index

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_plot[df_plot['Metric'] == 'OutlierRatio'], x='I_bin', y='Value', hue='Method', palette=method_palette, marker='o', errorbar=None, linewidth=2.5)
plt.title('Outlier Ratio vs Temporal Squared Gap Length Index (I)', pad=15, fontweight='bold')
plt.xlabel(r'Temporal Squared Gap Length Index (I) = $\frac{1}{N^2}\sum l_j^2$', fontweight='bold')
plt.ylabel('Outlier Ratio', fontweight='bold')
plt.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()